# Milestone 4 - Task 4: Email Header Feature Extraction

**Owner:** John Graham  
**Objective:** Build a header-feature extractor that parses raw RFC822 email headers and derives phishing-relevant signals (sender domain, free-mail sender, Reply-To mismatch, Received hops, DKIM/SPF presence, subject signals).

### Important data limitation
The combined training dataset (`phishing_email.csv` / `emails_clean.csv`) stores only merged body text in `text_combined` — it has **no raw headers**, so SPF/DKIM/Reply-To cannot be extracted from it. The only header-bearing corpus available is the raw **Enron** dataset (`legit-emails/emails.csv`), which is **legitimate-only** and predates DKIM/SPF (2001).

Therefore this task **validates the extractor structurally** on the Enron corpus. Labeled header-based classification is deferred to a later milestone once real emails (with modern headers) are captured from the M3 Postfix mail server. Security headers (DKIM/SPF) will read 0 on Enron — that is expected for an old, legitimate-only corpus.

## Step 0 — Download the raw header corpus from S3
Run this once in a terminal (the file is ~1.4 GB):
```bash
mkdir -p data/raw/legit-emails
aws s3 cp s3://email-security-pipeline-datasets/datasets/legit-emails/emails.csv \
  data/raw/legit-emails/emails.csv --profile lab-user
```

In [ ]:
from pathlib import Path
import csv, re
from email.parser import Parser
from email.utils import parseaddr, getaddresses
from itertools import islice
from collections import Counter

# csv has very large fields in this corpus
csv.field_size_limit(10_000_000)

## Step 1 — Configuration
We process a sample (not all ~517k rows) so the notebook runs quickly for screenshots. Increase `SAMPLE_SIZE` if you want a larger run.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT_PATH  = PROJECT_ROOT / 'data' / 'raw' / 'legit-emails' / 'emails.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'header_features_demo.csv'
SAMPLE_SIZE = 5000

FREEMAIL = {'gmail.com','yahoo.com','hotmail.com','outlook.com','aol.com','icloud.com'}
print('Input :', INPUT_PATH)
print('Output:', OUTPUT_PATH)

## Step 2 — Header parser
Each Enron row stores a full raw email in the `message` field. We parse it with Python's standard `email` library and pull out the header fields.

In [ ]:
def parse_headers(raw_message):
    """Parse a raw RFC822 message string into a header object."""
    return Parser().parsestr(str(raw_message))

def domain_of(addr):
    """Return the domain part of an email address, lowercased."""
    _, email_addr = parseaddr(addr or '')
    return email_addr.split('@')[-1].lower() if '@' in email_addr else ''

## Step 3 — Subject urgency cue (hardened, auxiliary)
The subject urgency check uses **stem patterns + `\b` word boundaries** rather than an exact-word list, so it generalizes over morphological variants (`verif` -> verify/verified/verifying) and avoids substring false positives (`limit` no longer fires on *unlimited*). As in M4-T6, this is only a small interpretable auxiliary signal — the primary phishing-language signal is TF-IDF over the email body at modeling time (M5).

In [ ]:
URGENCY_STEMS = [
    r'urgent', r'immediat', r'verif', r'suspend', r'limit', r'restrict',
    r'confirm', r'account', r'password', r'updat', r'expir',
    r'security alert', r'hurry', r'act now', r'last chance', r'time.?sensitive',
]
URGENCY_RE = re.compile(r'\b(?:' + '|'.join(URGENCY_STEMS) + r')', re.IGNORECASE)

def subject_has_urgency(subject):
    return int(bool(URGENCY_RE.search(str(subject))))

## Step 4 — Feature extractor
Nine header-derived features. Structural features (free-mail sender, domain match, subject signals) work on any corpus; security features (DKIM/SPF/Reply-To) read 0 on the old Enron data, as expected.

In [ ]:
def header_features(msg):
    from_dom = domain_of(msg.get('From'))
    to_addrs = [a for _, a in getaddresses(msg.get_all('To', []))]
    to_dom   = domain_of(to_addrs[0]) if to_addrs else ''
    reply_to = msg.get('Reply-To')
    reply_dom = domain_of(reply_to)
    subject  = str(msg.get('Subject') or '')
    received = msg.get_all('Received', []) or []
    return {
        'sender_freemail':        int(from_dom in FREEMAIL),
        'sender_recipient_match': int(bool(from_dom) and from_dom == to_dom),
        'has_reply_to':           int(reply_to is not None),
        'reply_to_mismatch':      int(bool(reply_dom) and reply_dom != from_dom),
        'num_received_hops':      len(received),
        'has_dkim':               int(msg.get('DKIM-Signature') is not None),
        'has_spf':                int(msg.get('Received-SPF') is not None or
                                      msg.get('Authentication-Results') is not None),
        'subject_length':         len(subject),
        'subject_has_urgency':    subject_has_urgency(subject),
    }

## Step 5 — Apply to a sample of the corpus

In [ ]:
rows_out = []
with INPUT_PATH.open('r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in islice(reader, SAMPLE_SIZE):
        msg = parse_headers(row['message'])
        rows_out.append(header_features(msg))

print(f'Processed {len(rows_out)} emails')
print('\nFeature columns:', list(rows_out[0].keys()))
print('\nSample row:', rows_out[0])

## Step 6 — Quick feature summary

In [ ]:
for col in rows_out[0].keys():
    vals = [r[col] for r in rows_out]
    if col in ('num_received_hops','subject_length'):
        print(f'{col:24s} avg={sum(vals)/len(vals):.1f}')
    else:
        print(f'{col:24s} positives={sum(1 for v in vals if v)}/{len(vals)}')

## Step 7 — Save the demonstration feature matrix

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cols = list(rows_out[0].keys())
with OUTPUT_PATH.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader(); w.writerows(rows_out)
print(f'Saved {len(rows_out)} rows -> {OUTPUT_PATH}')

## Conclusion
The header-feature extractor parses raw RFC822 headers and produces 9 phishing-relevant features. It is validated structurally on the Enron corpus. Security-header features (DKIM/SPF/Reply-To) are absent in this legacy legitimate-only dataset and will become meaningful once labeled emails are captured from the live M3 Postfix server in a future milestone.